# 01. Data Quality Assessment and Cleaning

This notebook prepares the UCI Online Retail transactions for customer growth, retention, and RFM analysis. The cleaning rules are explicit so that every excluded record can be explained and audited.

## 1. Import libraries and locate the raw file

By default, place the original file in `data/raw/Online Retail.xlsx`. For testing or another local location, set the `DATA_PATH` environment variable.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

default_path = repo_root / "data" / "raw" / "Online Retail.xlsx"
data_path = Path(os.environ.get("DATA_PATH", default_path))
processed_dir = repo_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"Reading data from: {data_path}")
if not data_path.exists():
    raise FileNotFoundError(
        "Online Retail.xlsx was not found. Follow data/README.md or set DATA_PATH."
    )

## 2. Load and understand the data

In [ ]:
raw = pd.read_excel(data_path)
print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
display(raw.head())
raw.info()

### Field definitions

| Field | Meaning |
|---|---|
| InvoiceNo | Order/invoice identifier; values beginning with `C` are cancellations |
| StockCode | Product identifier |
| Description | Product description |
| Quantity | Units in the transaction line |
| InvoiceDate | Transaction timestamp |
| UnitPrice | Price per unit in GBP |
| CustomerID | Customer identifier |
| Country | Customer country |

## 3. Build a data-quality report

In [ ]:
quality_report = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_pct": raw.isna().mean(),
    "unique_values": raw.nunique(dropna=True),
})
quality_report["missing_pct"] = quality_report["missing_pct"].map(lambda x: f"{x:.2%}")
display(quality_report)

raw_flags = pd.Series({
    "rows": len(raw),
    "duplicate_rows": raw.duplicated().sum(),
    "missing_customer_id": raw["CustomerID"].isna().sum(),
    "cancelled_invoice_rows": raw["InvoiceNo"].astype(str).str.upper().str.startswith("C").sum(),
    "nonpositive_quantity_rows": (raw["Quantity"] <= 0).sum(),
    "nonpositive_price_rows": (raw["UnitPrice"] <= 0).sum(),
})
display(raw_flags.to_frame("count"))

## 4. Apply transparent cleaning rules

For customer-level growth and retention analysis, this notebook:

1. removes exact duplicate transaction lines;
2. excludes rows without a customer identifier because they cannot be assigned to a cohort;
3. excludes cancellation invoices;
4. keeps only positive quantities and prices;
5. creates `Revenue = Quantity × UnitPrice`.

Cancellations are measured before exclusion and can be analyzed separately later.

In [ ]:
df = raw.copy()
audit = [{"step": "Raw data", "rows_remaining": len(df), "rows_removed": 0}]

def apply_filter(frame, mask, step):
    before = len(frame)
    result = frame.loc[mask].copy()
    audit.append({
        "step": step,
        "rows_remaining": len(result),
        "rows_removed": before - len(result),
    })
    return result

before = len(df)
df = df.drop_duplicates().copy()
audit.append({
    "step": "Remove exact duplicates",
    "rows_remaining": len(df),
    "rows_removed": before - len(df),
})

df = apply_filter(df, df["CustomerID"].notna(), "Exclude missing CustomerID")
df = apply_filter(
    df,
    ~df["InvoiceNo"].astype(str).str.upper().str.startswith("C"),
    "Exclude cancellation invoices",
)
df = apply_filter(df, df["Quantity"] > 0, "Keep positive quantity")
df = apply_filter(df, df["UnitPrice"] > 0, "Keep positive unit price")

df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
df["CustomerID"] = df["CustomerID"].astype("int64").astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

audit_df = pd.DataFrame(audit)
audit_df["retention_pct_of_raw"] = audit_df["rows_remaining"] / len(raw)
display(audit_df)

## 5. Validate the cleaned dataset

In [ ]:
assert not df.duplicated().any(), "Duplicate rows remain."
assert df["CustomerID"].notna().all(), "CustomerID still contains missing values."
assert (df["Quantity"] > 0).all(), "Nonpositive quantities remain."
assert (df["UnitPrice"] > 0).all(), "Nonpositive prices remain."
assert (df["Revenue"] > 0).all(), "Nonpositive revenue remains."
assert not df["InvoiceNo"].str.upper().str.startswith("C").any(), "Cancellations remain."

clean_summary = pd.Series({
    "transaction_lines": len(df),
    "orders": df["InvoiceNo"].nunique(),
    "customers": df["CustomerID"].nunique(),
    "products": df["StockCode"].nunique(),
    "countries": df["Country"].nunique(),
    "start_date": df["InvoiceDate"].min(),
    "end_date": df["InvoiceDate"].max(),
    "revenue_gbp": df["Revenue"].sum(),
})
display(clean_summary.to_frame("value"))
display(df.head())

## 6. Export the reproducible analysis table

In [ ]:
output_path = processed_dir / "online_retail_clean.csv.gz"
df.to_csv(output_path, index=False, compression="gzip")
print(f"Saved {len(df):,} cleaned rows to: {output_path}")

## Key takeaway

The cleaned table is suitable for customer-level KPI, cohort retention, and RFM analysis. Anonymous transactions and cancellations were not treated as normal purchases; their removal is documented in the audit table so the analytical population remains transparent.